In [5]:
import json
import hashlib
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [6]:
# Configuration
DATA_PATH = Path('selfplay_guided_1000games_v2.jsonl')
OUTPUT_DIR = Path('catan_winner_model_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SPLIT_SEED = 'guided_turn_winner_v1'

# To prevent very long games from dominating, keep at most this many rows per game.
# Set to None to keep all rows.
MAX_ROWS_PER_GAME: Optional[int] = 128

# Optional stage-aware reporting weights, used only for reporting, not training.
STAGE_WEIGHTS = {'early': 0.5, 'mid': 0.3, 'late': 0.2}

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9
print(DATA_PATH)
print(OUTPUT_DIR)


selfplay_guided_1000games_v2.jsonl
catan_winner_model_outputs


In [9]:
def deterministic_split_for_game(game_id: str, split_seed: str = SPLIT_SEED) -> str:
    digest = hashlib.md5(f'{split_seed}:{game_id}'.encode('utf-8')).hexdigest()
    bucket = int(digest[:8], 16) / float(16**8)
    if bucket < TRAIN_RATIO:
        return 'train'
    if bucket < TRAIN_RATIO + VAL_RATIO:
        return 'val'
    return 'test'


def infer_stage(turn_index: int) -> str:
    if turn_index < 80:
        return 'early'
    if turn_index < 180:
        return 'mid'
    return 'late'


def parse_action_features(action_taken: str) -> Dict[str, Any]:
    action_taken = action_taken or ''
    actions = [a.strip() for a in action_taken.split('+') if a.strip()]
    action_set = set(actions)
    features = {
        'action_raw': action_taken,
        'action_count': len(actions),
        'has_pass': int('pass' in action_set),
        'has_trade_bank': int('trade_bank' in action_set),
        'has_build_road': int('build_road' in action_set),
        'has_build_settlement': int('build_settlement' in action_set),
        'has_build_city': int('build_city' in action_set),
        'has_buy_dev_card': int('buy_dev_card' in action_set),
    }
    return features


def flatten_row(row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if row.get('termination_reason') != 'winner':
        return None

    turn_meta = row.get('turn_metadata', {}) or {}
    game_id = row.get('game_id')
    winner_name = row.get('winner_player_name')
    current_player_name = turn_meta.get('current_player_name')

    if not game_id or not winner_name or not current_player_name:
        return None

    player_state = row.get('player_state', []) or []
    player_lookup = {}
    ordered_names = []
    for player in player_state:
        if not isinstance(player, dict):
            continue
        pname = str(player.get('player_name'))
        ordered_names.append(pname)
        player_lookup[pname] = player

    if current_player_name not in player_lookup:
        return None

    others = [name for name in ordered_names if name != current_player_name]

    out: Dict[str, Any] = {
        'game_id': str(game_id),
        'position_id': row.get('position_id'),
        'row_index_in_game': row.get('row_index_in_game'),
        'turn_index': int(turn_meta.get('turn_index', -1)),
        'current_player_name': str(current_player_name),
        'current_player_id': str(turn_meta.get('current_player_id')) if turn_meta.get('current_player_id') is not None else '',
        'current_phase': str(turn_meta.get('current_phase') or 'unknown'),
        'roll': int(row.get('roll', -1)) if row.get('roll') is not None else -1,
        'is_terminal': int(bool(row.get('is_terminal', False))),
        'winner_player_name': str(winner_name),
        'y_win': int(current_player_name == winner_name),
        'termination_reason': str(row.get('termination_reason') or ''),
        'simulation_error': str(row.get('simulation_error') or ''),
    }

    out['stage'] = infer_stage(out['turn_index']) if out['turn_index'] >= 0 else 'unknown'

    current_player = player_lookup[current_player_name]
    current_resources = current_player.get('resources', {}) or {}
    out['self_victory_points'] = int(current_player.get('victory_points', 0))
    for res in ['BRICK', 'WOOL', 'ORE', 'GRAIN', 'LUMBER']:
        out[f'self_res_{res.lower()}'] = int(current_resources.get(res, 0))
    out['self_total_resources'] = sum(out[f'self_res_{res.lower()}'] for res in ['BRICK', 'WOOL', 'ORE', 'GRAIN', 'LUMBER'])

    opp_vps = []
    opp_totals = []
    for idx in range(3):
        if idx < len(others):
            opp = player_lookup[others[idx]]
            opp_res = opp.get('resources', {}) or {}
            out[f'opp{idx+1}_name'] = others[idx]
            out[f'opp{idx+1}_victory_points'] = int(opp.get('victory_points', 0))
            opp_vps.append(out[f'opp{idx+1}_victory_points'])
            total = 0
            for res in ['BRICK', 'WOOL', 'ORE', 'GRAIN', 'LUMBER']:
                value = int(opp_res.get(res, 0))
                out[f'opp{idx+1}_res_{res.lower()}'] = value
                total += value
            out[f'opp{idx+1}_total_resources'] = total
            opp_totals.append(total)
        else:
            out[f'opp{idx+1}_name'] = ''
            out[f'opp{idx+1}_victory_points'] = 0
            for res in ['BRICK', 'WOOL', 'ORE', 'GRAIN', 'LUMBER']:
                out[f'opp{idx+1}_res_{res.lower()}'] = 0
            out[f'opp{idx+1}_total_resources'] = 0
            opp_vps.append(0)
            opp_totals.append(0)

    out['best_opp_victory_points'] = max(opp_vps) if opp_vps else 0
    out['vp_lead_vs_best_opp'] = out['self_victory_points'] - out['best_opp_victory_points']
    out['best_opp_total_resources'] = max(opp_totals) if opp_totals else 0
    out['resource_lead_vs_best_opp'] = out['self_total_resources'] - out['best_opp_total_resources']

    out.update(parse_action_features(str(row.get('action_taken') or '')))
    return out


def load_jsonl_rows(path: Path, skip_bad_lines: bool = True, max_bad_lines_to_print: int = 10) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    bad_lines = 0

    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                bad_lines += 1
                if bad_lines <= max_bad_lines_to_print:
                    print(f"Bad JSONL line {line_num}: {exc}")
                    print(line[:300])
                    print("-" * 80)

                if skip_bad_lines:
                    continue
                raise ValueError(f"Invalid JSONL at line {line_num}: {exc}") from exc

            if isinstance(obj, dict):
                rows.append(obj)

    print(f"Loaded {len(rows)} valid rows")
    print(f"Skipped {bad_lines} malformed rows")
    return rows

In [10]:
raw_rows = load_jsonl_rows(DATA_PATH)
print('Raw rows loaded:', len(raw_rows))

flat_rows = []
for row in raw_rows:
    flat = flatten_row(row)
    if flat is not None:
        flat_rows.append(flat)

df = pd.DataFrame(flat_rows)
print('Winner-only flattened rows:', len(df))
print('Unique games:', df['game_id'].nunique())
df.head(3)

Bad JSONL line 538254: Unterminated string starting at: line 1 column 393 (char 392)
{"game_id":"4a0250b4-c765-4829-9b87-aa3bb82cacdc","position_id":"4a0250b4-c765-4829-9b87-aa3bb82cacdc_593","row_index_in_game":593,"turn_metadata":{"turn_index":593,"current_player_id":"1","current_player_name":"P2","current_phase":"main"},"action_taken":"pass","roll":8,"player_state":[{"player_name
--------------------------------------------------------------------------------
Loaded 538253 valid rows
Skipped 1 malformed rows
Raw rows loaded: 538253
Winner-only flattened rows: 210060
Unique games: 531


,game_id,position_id,row_index_in_game,turn_index,current_player_name,current_player_id,current_phase,roll,is_terminal,winner_player_name,...,best_opp_total_resources,resource_lead_vs_best_opp,action_raw,action_count,has_pass,has_trade_bank,has_build_road,has_build_settlement,has_build_city,has_buy_dev_card
0,1b455721-8576-484a-b9ca-b7b2c57fe23c,1b455721-8576-484a-b9ca-b7b2c57fe23c_0,0,0,P2,0,main,6,0,P1,...,2,-2,pass,1,1,0,0,0,0,0
1,1b455721-8576-484a-b9ca-b7b2c57fe23c,1b455721-8576-484a-b9ca-b7b2c57fe23c_1,1,1,P0,1,main,7,0,P1,...,2,-2,pass,1,1,0,0,0,0,0
2,1b455721-8576-484a-b9ca-b7b2c57fe23c,1b455721-8576-484a-b9ca-b7b2c57fe23c_2,2,2,P1,2,main,7,0,P1,...,0,2,pass,1,1,0,0,0,0,0


In [11]:
# Basic dataset QA
summary = {
    'rows': int(len(df)),
    'unique_games': int(df['game_id'].nunique()),
    'class_balance': df['y_win'].value_counts(dropna=False).to_dict(),
    'stage_balance': df['stage'].value_counts(dropna=False).to_dict(),
    'phase_balance': df['current_phase'].value_counts(dropna=False).to_dict(),
    'action_count_balance': df['action_count'].value_counts(dropna=False).sort_index().to_dict(),
}
summary

{'rows': 210060,
 'unique_games': 531,
 'class_balance': {0: 157413, 1: 52647},
 'stage_balance': {'late': 116025, 'mid': 51555, 'early': 42480},
 'phase_balance': {'main': 210060},
 'action_count_balance': {1: 146350, 2: 49457, 3: 11763, 4: 2490}}

In [12]:
# Optional cap on rows per game so very long games do not dominate.
if MAX_ROWS_PER_GAME is not None:
    df = (
        df.sort_values(['game_id', 'row_index_in_game'])
          .groupby('game_id', group_keys=False)
          .head(MAX_ROWS_PER_GAME)
          .reset_index(drop=True)
    )

print('Rows after per-game cap:', len(df))
print('Unique games after cap:', df['game_id'].nunique())


Rows after per-game cap: 67905
Unique games after cap: 531


In [13]:
# Deterministic game-level split
game_ids = sorted(df['game_id'].unique())
split_manifest = {gid: deterministic_split_for_game(gid) for gid in game_ids}
df['split'] = df['game_id'].map(split_manifest)

split_summary = {
    split: {
        'rows': int((df['split'] == split).sum()),
        'games': int(sum(1 for gid, s in split_manifest.items() if s == split)),
        'class_balance': df.loc[df['split'] == split, 'y_win'].value_counts(dropna=False).to_dict(),
    }
    for split in ['train', 'val', 'test']
}
split_summary

{'train': {'rows': 54593, 'games': 427, 'class_balance': {0: 40943, 1: 13650}},
 'val': {'rows': 6400, 'games': 50, 'class_balance': {0: 4800, 1: 1600}},
 'test': {'rows': 6912, 'games': 54, 'class_balance': {0: 5184, 1: 1728}}}

In [14]:
# Select features
drop_cols = {
    'game_id', 'position_id', 'winner_player_name', 'y_win', 'split', 'termination_reason', 'simulation_error'
}
feature_cols = [c for c in df.columns if c not in drop_cols]

categorical_cols = [
    c for c in feature_cols
    if df[c].dtype == 'object'
]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print('Feature count:', len(feature_cols))
print('Numeric count:', len(numeric_cols))
print('Categorical count:', len(categorical_cols))
print('Sample categorical cols:', categorical_cols[:10])


Feature count: 51
Numeric count: 43
Categorical count: 8
Sample categorical cols: ['current_player_name', 'current_player_id', 'current_phase', 'stage', 'opp1_name', 'opp2_name', 'opp3_name', 'action_raw']


In [15]:
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train = train_df[feature_cols]
y_train = train_df['y_win'].astype(int)
X_val = val_df[feature_cols]
y_val = val_df['y_win'].astype(int)
X_test = test_df[feature_cols]
y_test = test_df['y_win'].astype(int)

preprocessor_linear = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

# HistGradientBoosting needs dense numeric features.
X_train_hgb = pd.get_dummies(X_train, columns=categorical_cols, dummy_na=True)
X_val_hgb = pd.get_dummies(X_val, columns=categorical_cols, dummy_na=True)
X_test_hgb = pd.get_dummies(X_test, columns=categorical_cols, dummy_na=True)

X_val_hgb = X_val_hgb.reindex(columns=X_train_hgb.columns, fill_value=0)
X_test_hgb = X_test_hgb.reindex(columns=X_train_hgb.columns, fill_value=0)

X_train_hgb = X_train_hgb.fillna(0)
X_val_hgb = X_val_hgb.fillna(0)
X_test_hgb = X_test_hgb.fillna(0)


In [16]:
def compute_metrics(y_true, proba, pred) -> Dict[str, float]:
    metrics = {
        'log_loss': float(log_loss(y_true, proba, labels=[0, 1])),
        'accuracy': float(accuracy_score(y_true, pred)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
    }
    if len(set(y_true)) > 1:
        metrics['roc_auc'] = float(roc_auc_score(y_true, proba))
        metrics['pr_auc'] = float(average_precision_score(y_true, proba))
    else:
        metrics['roc_auc'] = float('nan')
        metrics['pr_auc'] = float('nan')
    return metrics


def compute_stage_metrics(split_df: pd.DataFrame, probas: np.ndarray, preds: np.ndarray) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    weighted_log_loss = 0.0
    total_weight = 0.0
    for stage, weight in STAGE_WEIGHTS.items():
        mask = split_df['stage'] == stage
        if mask.sum() == 0:
            continue
        y = split_df.loc[mask, 'y_win'].astype(int).to_numpy()
        p = probas[mask.to_numpy()]
        pr = preds[mask.to_numpy()]
        stage_metrics = compute_metrics(y, p, pr)
        stage_metrics['rows'] = int(mask.sum())
        out[stage] = stage_metrics
        weighted_log_loss += weight * stage_metrics['log_loss']
        total_weight += weight
    out['stage_balanced_log_loss'] = float(weighted_log_loss / total_weight) if total_weight > 0 else float('nan')
    return out


In [17]:
# Train baseline models
models = {}
results = {}

# Majority baseline
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_hgb, y_train)
dummy_val_proba = dummy.predict_proba(X_val_hgb)[:, 1]
dummy_val_pred = dummy.predict(X_val_hgb)
results['majority'] = {
    'val': compute_metrics(y_val, dummy_val_proba, dummy_val_pred),
    'val_stage_metrics': compute_stage_metrics(val_df, dummy_val_proba, dummy_val_pred),
}
models['majority'] = dummy

# Logistic regression baseline
logreg = Pipeline([
    ('preprocess', preprocessor_linear),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=None, random_state=42)),
])
logreg.fit(X_train, y_train)
logreg_val_proba = logreg.predict_proba(X_val)[:, 1]
logreg_val_pred = (logreg_val_proba >= 0.5).astype(int)
results['logreg'] = {
    'val': compute_metrics(y_val, logreg_val_proba, logreg_val_pred),
    'val_stage_metrics': compute_stage_metrics(val_df, logreg_val_proba, logreg_val_pred),
}
models['logreg'] = logreg

# Histogram Gradient Boosting
hgb = HistGradientBoostingClassifier(
    max_depth=8,
    learning_rate=0.05,
    max_iter=300,
    min_samples_leaf=50,
    l2_regularization=0.0,
    random_state=42,
)
hgb.fit(X_train_hgb, y_train)
hgb_val_proba = hgb.predict_proba(X_val_hgb)[:, 1]
hgb_val_pred = (hgb_val_proba >= 0.5).astype(int)
results['hgb'] = {
    'val': compute_metrics(y_val, hgb_val_proba, hgb_val_pred),
    'val_stage_metrics': compute_stage_metrics(val_df, hgb_val_proba, hgb_val_pred),
}
models['hgb'] = hgb

results

{'majority': {'val': {'log_loss': 9.010913347279287,
   'accuracy': 0.75,
   'f1': 0.0,
   'roc_auc': 0.5,
   'pr_auc': 0.25},
  'val_stage_metrics': {'early': {'log_loss': 9.010913347279285,
    'accuracy': 0.75,
    'f1': 0.0,
    'roc_auc': 0.5,
    'pr_auc': 0.25,
    'rows': 4000},
   'mid': {'log_loss': 9.010913347279287,
    'accuracy': 0.75,
    'f1': 0.0,
    'roc_auc': 0.5,
    'pr_auc': 0.25,
    'rows': 2400},
   'stage_balanced_log_loss': 9.010913347279285}},
 'logreg': {'val': {'log_loss': 0.471892996518647,
   'accuracy': 0.744375,
   'f1': 0.6172204024333178,
   'roc_auc': 0.8579217447916666,
   'pr_auc': 0.6483833255474287},
  'val_stage_metrics': {'early': {'log_loss': 0.4976298065143797,
    'accuracy': 0.70325,
    'f1': 0.5836548579445808,
    'roc_auc': 0.83345,
    'pr_auc': 0.5997072031530406,
    'rows': 4000},
   'mid': {'log_loss': 0.42899831319242576,
    'accuracy': 0.8129166666666666,
    'f1': 0.6844694307800422,
    'roc_auc': 0.8905453703703703,
    'pr

In [18]:
def model_score(name: str) -> float:
    stage_score = results[name]['val_stage_metrics'].get('stage_balanced_log_loss', np.nan)
    if pd.isna(stage_score):
        return results[name]['val']['log_loss']
    return stage_score

best_model_name = min(['majority', 'logreg', 'hgb'], key=model_score)
best_model_name

'hgb'

In [19]:
# Final evaluation on test set
final_metrics = {
    'selected_model': best_model_name,
    'validation_results': results,
    'test': {},
}

for name, model in models.items():
    if name == 'majority':
        proba = model.predict_proba(X_test_hgb)[:, 1]
        pred = model.predict(X_test_hgb)
    elif name == 'logreg':
        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
    elif name == 'hgb':
        proba = model.predict_proba(X_test_hgb)[:, 1]
        pred = (proba >= 0.5).astype(int)
    else:
        continue

    final_metrics['test'][name] = {
        'overall': compute_metrics(y_test, proba, pred),
        'stage_metrics': compute_stage_metrics(test_df, proba, pred),
    }

final_metrics

{'selected_model': 'hgb',
 'validation_results': {'majority': {'val': {'log_loss': 9.010913347279287,
    'accuracy': 0.75,
    'f1': 0.0,
    'roc_auc': 0.5,
    'pr_auc': 0.25},
   'val_stage_metrics': {'early': {'log_loss': 9.010913347279285,
     'accuracy': 0.75,
     'f1': 0.0,
     'roc_auc': 0.5,
     'pr_auc': 0.25,
     'rows': 4000},
    'mid': {'log_loss': 9.010913347279287,
     'accuracy': 0.75,
     'f1': 0.0,
     'roc_auc': 0.5,
     'pr_auc': 0.25,
     'rows': 2400},
    'stage_balanced_log_loss': 9.010913347279285}},
  'logreg': {'val': {'log_loss': 0.471892996518647,
    'accuracy': 0.744375,
    'f1': 0.6172204024333178,
    'roc_auc': 0.8579217447916666,
    'pr_auc': 0.6483833255474287},
   'val_stage_metrics': {'early': {'log_loss': 0.4976298065143797,
     'accuracy': 0.70325,
     'f1': 0.5836548579445808,
     'roc_auc': 0.83345,
     'pr_auc': 0.5997072031530406,
     'rows': 4000},
    'mid': {'log_loss': 0.42899831319242576,
     'accuracy': 0.81291666666

In [20]:
# Save outputs
metrics_path = OUTPUT_DIR / 'metrics.json'
split_summary_path = OUTPUT_DIR / 'split_summary.json'
model_bundle_path = OUTPUT_DIR / f'{best_model_name}_winner_model_bundle.joblib'
feature_columns_path = OUTPUT_DIR / 'feature_columns.json'
config_path = OUTPUT_DIR / 'config.json'

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(final_metrics, f, indent=2)

with split_summary_path.open('w', encoding='utf-8') as f:
    json.dump(split_summary, f, indent=2)

with feature_columns_path.open('w', encoding='utf-8') as f:
    json.dump({
        'feature_cols': feature_cols,
        'numeric_cols': numeric_cols,
        'categorical_cols': categorical_cols,
        'hgb_columns': X_train_hgb.columns.tolist(),
    }, f, indent=2)

with config_path.open('w', encoding='utf-8') as f:
    json.dump({
        'data_path': str(DATA_PATH),
        'max_rows_per_game': MAX_ROWS_PER_GAME,
        'split_seed': SPLIT_SEED,
        'stage_weights': STAGE_WEIGHTS,
        'selected_model': best_model_name,
    }, f, indent=2)

bundle = {
    'model_name': best_model_name,
    'model': models[best_model_name],
    'feature_cols': feature_cols,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'hgb_columns': X_train_hgb.columns.tolist(),
    'config': {
        'split_seed': SPLIT_SEED,
        'max_rows_per_game': MAX_ROWS_PER_GAME,
        'stage_weights': STAGE_WEIGHTS,
    },
}
joblib.dump(bundle, model_bundle_path)

print('Saved:')
print(metrics_path)
print(split_summary_path)
print(feature_columns_path)
print(config_path)
print(model_bundle_path)


Saved:
catan_winner_model_outputs/metrics.json
catan_winner_model_outputs/split_summary.json
catan_winner_model_outputs/feature_columns.json
catan_winner_model_outputs/config.json
catan_winner_model_outputs/hgb_winner_model_bundle.joblib
